In [3]:
from pulp import *
from typing import Dict, Any



In [4]:
prob = LpProblem("Problem_1", LpMaximize)

#decision variables
x = LpVariable.dicts("x", [1,2], lowBound=0, cat='Continuous')

#objective function
prob += 10*x[1] + 20*x[2]
prob += -x[1] + 2*x[2] <= 15
prob += x[1] + x[2] <= 12
prob += 5*x[1] + 3*x[2] <= 45

print(x)
print(prob)

status = prob.solve()
print(LpStatus[status])

for var in prob.variables():
    print(f"{var.name} = {var.varValue}")

{1: x_1, 2: x_2}
Problem_1:
MAXIMIZE
10*x_1 + 20*x_2 + 0
SUBJECT TO
_C1: - x_1 + 2 x_2 <= 15

_C2: x_1 + x_2 <= 12

_C3: 5 x_1 + 3 x_2 <= 45

VARIABLES
x_1 Continuous
x_2 Continuous

Optimal
x_1 = 3.0
x_2 = 9.0


In [ ]:

BOX_TYPES = {
"Standard_Medium": {"volume_cap": 1200, "weight_cap": 25, "base_cost": 3.50},
"Standard_Large":  {"volume_cap": 2200, "weight_cap": 45, "base_cost": 5.00}
}

ITEMS = {
    "Item1_Organic_Milk":     {"volume": 120, "weight": 8.6,  "fragile": 0},
    "Item2_Almond_Milk":      {"volume": 120, "weight": 8.6,  "fragile": 0},
    "Item3_Fresh_Spinach":    {"volume": 180, "weight": 0.5,  "fragile": 1},
    "Item4_Avocados_4ct":     {"volume": 80,  "weight": 1.2,  "fragile": 1},
    "Item5_Sweet_Potatoes":   {"volume": 150, "weight": 5.0,  "fragile": 0},
    "Item6_Salmon_Fillets":   {"volume": 60,  "weight": 1.5,  "fragile": 0},
    "Item7_Greek_Yogurt_Tub": {"volume": 90,  "weight": 2.0,  "fragile": 0},
    "Item8_Sprouted_Bread":   {"volume": 200, "weight": 1.3,  "fragile": 1},
    "Item9_Baby_Carrots":     {"volume": 70,  "weight": 1.0,  "fragile": 0},
    "Item10_Grass_Fed_Beef":  {"volume": 50,  "weight": 1.0,  "fragile": 0},
}
    
    # 1. SETUP CANDIDATE BOX POOL
    # For MVP, assume max allowance of 3 of each box type can be generated
MAX_COPIES = 3
box_pool = []
for b_type, attrs in BOX_TYPES.items():
    for i in range(MAX_COPIES):
        box_pool.append({
            "id": f"{b_type}_{i}",
            "type": b_type,
            **attrs
        })
box_ids = [box["id"] for box in box_pool]
item_ids = list(ITEMS.keys())

heavy_items = [item for item, attrs in ITEMS.items() if attrs["weight"] > 4.0]
fragile_items = [item for item, attrs in ITEMS.items() if attrs["fragile"] == 1]
    # --- YOUR WORK GOES HERE ---
    # TODO: Define decision variables using pulp.LpVariable.dicts
prob = pulp.LpProblem("Dinner_Box_Packing_Optimization", LpMinimize)
x = pulp.LpVariable.dicts("x", (box_ids), cat="Binary")
y = pulp.LpVariable.dicts("y", (item_ids, box_ids), cat="Binary")
w = pulp.LpVariable.dicts("box_type_fragile", (box_ids), cat="Binary")

    # TODO: Implement Objective Function (Minimize Cost)
prob += pulp.lpSum([b["base_cost"] * x[b["id"]] for b in box_pool])
# TODO: Implement Constraints (Assignment, Caps, Big-M Crushing Rule)
for i in item_ids:
    prob += pulp.lpSum([y[i][bid] for bid in box_ids]) == 1

for b in box_pool:
    bid = b['id']
    prob += pulp.lpSum([ITEMS[i]["volume"] * y[i][bid] for i in item_ids]) <= b["volume_cap"] * x[bid]
    prob += pulp.lpSum([ITEMS[i]["weight"] * y[i][bid] for i in item_ids]) <= b["weight_cap"] * x[bid]

for b in box_pool:
    bid = b['id']
    prob += pulp.lpSum([y[i][bid] for i in fragile_items]) <= len(fragile_items) * w[bid]
    prob += pulp.lpSum([y[i][bid] for i in heavy_items]) <= len(heavy_items) * (1 - w[bid])
for b in box_pool:
    bid = b['id']
    prob += w[bid] <= x[bid] , f"Bind_fragile_switch_{bid}"
    
    
    # Dummy placeholder for structural validation:

status = prob.solve()
print(LpStatus[status])

for b in box_pool:
    bid = b["id"]
    assigned_items = [i for i in item_ids if pulp.value(y[i][bid]) == 1]
    if assigned_items:
        print(f"📦 Box: {bid} ({b['type']})")
        print(f"   Items packed:")
        for item in assigned_items:
            print(f"      - {item}")
        print("-" * 40)

Optimal
📦 Box: Standard_Medium_0 (Standard_Medium)
   Items packed:
      - Item3_Fresh_Spinach
      - Item4_Avocados_4ct
      - Item6_Salmon_Fillets
      - Item7_Greek_Yogurt_Tub
      - Item8_Sprouted_Bread
      - Item9_Baby_Carrots
----------------------------------------
📦 Box: Standard_Medium_2 (Standard_Medium)
   Items packed:
      - Item1_Organic_Milk
      - Item2_Almond_Milk
      - Item5_Sweet_Potatoes
      - Item10_Grass_Fed_Beef
----------------------------------------
